# Training SFT (QLoRA) — CoT vs non-CoT
Latih student `Qwen2.5-1.5B`/`0.5B` dari `data/sft/cot.jsonl` & `nocot.jsonl`.
Hyperparam identik, beda **cuma dataset** -> isolasi efek CoT.

CoT digenerate **per-dataset** (numglue/aimo_hard/easy). Cell §3 **gabung** ketiganya jadi 1 set per arm -> 2 model (cot, nocot), bukan per dataset.

**Setting Kaggle:** GPU **T4 x2** (1 GPU cukup utk 1.5B QLoRA), Internet **On**.
Add Input = dataset hasil CoT (berisi folder `sft/<ds>/cot.jsonl` + `nocot.jsonl`).

### 1. Install

In [ ]:
!pip install -q -U unsloth unsloth_zoo
!pip install -q -U datasets pyyaml

### 2. Clone/pull repo (permanen)
Repo private -> butuh PAT. Simpan Kaggle Secret `GH_TOKEN` (fine-grained, Contents:Read).
Kalau repo sudah public, hapus bagian token.

In [ ]:
import os, sys, subprocess
from pathlib import Path
REPO = '/kaggle/working/FP_NLP'
TOKEN = ''
try:
    from kaggle_secrets import UserSecretsClient
    TOKEN = UserSecretsClient().get_secret('GH_TOKEN')
except Exception as e:
    print('no GH_TOKEN secret (ok kalau repo public):', e)
auth = f'{TOKEN}@' if TOKEN else ''
URL = f'https://{auth}github.com/henray404/FP_NLP.git'
if os.path.exists(REPO):
    subprocess.run(['git','-C',REPO,'pull','-q'], check=True)
else:
    subprocess.run(['git','clone','-q',URL,REPO], check=True)
sys.path.insert(0, REPO)
os.chdir(REPO)
print('repo ready:', REPO)

### 3. Merge dataset SFT (3 dataset -> 1 per arm)
CoT digenerate per-dataset (`sft/numglue/`, `sft/aimo_hard/`, `sft/easy/`). Cell ini **gabung** semua `cot.jsonl` jadi satu `data/sft/cot.jsonl`, dan semua `nocot.jsonl` jadi `data/sft/nocot.jsonl` -> 1 model per arm (bukan per dataset).

Cari di `/kaggle/input` (Add dataset hasil CoT). Dataset disjoint; dedup baris jaga-jaga.

In [ ]:
import glob
Path('data/sft').mkdir(parents=True, exist_ok=True)

def merge_arm(name):
    """Gabung semua sft/<ds>/<name> di /kaggle/input -> data/sft/<name> (dedup baris)."""
    hits = sorted(glob.glob(f'/kaggle/input/**/sft/*/{name}', recursive=True))
    if not hits:                                   # fallback: file tunggal di mana saja
        hits = sorted(glob.glob(f'/kaggle/input/**/{name}', recursive=True))
    seen, rows = set(), []
    for h in hits:
        for line in open(h, encoding='utf-8'):
            line = line.strip()
            if line and line not in seen:
                seen.add(line); rows.append(line)
    out = f'data/sft/{name}'
    with open(out, 'w', encoding='utf-8') as f:
        if rows:
            f.write('\n'.join(rows) + '\n')
    print(f'{name}: {len(hits)} file -> {len(rows)} baris -> {out}')
    for h in hits:
        print('   +', h)
    return len(rows)

n_cot = merge_arm('cot.jsonl')
n_nocot = merge_arm('nocot.jsonl')
assert n_cot and n_nocot, 'dataset SFT kosong -- Add dataset hasil CoT (folder sft/) ke notebook'
print(f'\nmerged: cot={n_cot} nocot={n_nocot}')

### 4. Konfigurasi run
Pilih config (1.5B default). `MAX_EXAMPLES` kecil dulu buat smoke test, lalu `None` utk full.

In [ ]:
from src.training.train_sft import load_config, build_and_train
CONFIGS = ['src/training/configs/cot_1.5b.yaml',
           'src/training/configs/nocot_1.5b.yaml']
MAX_EXAMPLES = 50   # None = full
OUT_ROOT = '/kaggle/working'

### 5. Train (CoT lalu non-CoT)

In [ ]:
import gc, torch
for cfg_path in CONFIGS:
    cfg = load_config(cfg_path, max_examples=MAX_EXAMPLES)
    cfg.output_dir = f"{OUT_ROOT}/adapter_{Path(cfg_path).stem}"
    print('==='*12, cfg.mode, cfg.base_model, '->', cfg.output_dir)
    build_and_train(cfg)
    gc.collect(); torch.cuda.empty_cache()

### 6. Zip adapter -> Output
Eval pakai `notebooks/skenario4_eval_kaggle.ipynb` (load base + adapter sebagai 'model kita').

In [ ]:
!cd /kaggle/working && zip -rq adapters_sft.zip adapter_* && echo 'adapters_sft.zip siap di Output'